In [10]:
import librosa
import IPython.display as ipd
import requests
import io
import re

from google.cloud import storage

import soundfile as sf
import wordninja

# Note
Mixer functions originally built for wav files but some of these are stored as flac / other types - may need to change mixer functions to sf.read() instead of wavfile.read()

# Step 1
Get URLs from event buckets of interest and store them in a dictionary that has a user-friendly key for the clip.
I need to expand this code to iterate through all folders with sound clips and not just this one but this is the idea:

In [4]:
bucket_name = "noaa-passive-bioacoustic"
prefix = "sanctsound/products/sound_clips/"

client = storage.Client.create_anonymous_client()
bucket = client.bucket(bucket_name)
blobs = client.list_blobs(bucket, prefix=prefix)

urls = {}

for blob in blobs:  
    if not blob.name.endswith('.wav'):
        continue

    match = re.search(r"SanctSound_OC03_02_([^_]+)_\d{8}T\d{6}Z\.wav$", blob.name)
    if not match:
        continue

    event_name = match.group(1)

    #if "whale" in event_name:
     #   species = event_name.split("whale") 
      #  formatted_name = ' '.join([p.capitalize() for p in species if p] + ['whale'])
    #else:
      #  formatted_name = event_name.capitalize()

    #formatted_name += " - SanctSound"

    urls[event_name] = bucket_name + "/" + blob.name


urls

{'humpbackwhale': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_humpbackwhale_20191104T151853Z.wav',
 'killerwhale': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_killerwhale_20191128T091339Z.wav',
 'ship': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_ship_20200228T080107Z.wav',
 'soundscape': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_soundscape_20191220T211558Z.wav'}

In [14]:
bucket_name = "noaa-passive-bioacoustic"
prefix = "sanctsound/products/sound_clips/"

client = storage.Client.create_anonymous_client()
bucket = client.bucket(bucket_name)
blobs = client.list_blobs(bucket, prefix=prefix)

urls = {}
keywords = ['shrimp', 'whale', 'ship', 'boat', 'dolphins', 'odontocete', 'wind', 'sealion', 'fish', 'ship', 'vessel', 'scuba', 'hurricane', 'pinniped', 'seal', 'bocaccio', 'sonar', 'rain']

for blob in blobs:
    if not blob.name.endswith('.wav'):
        continue

    match = re.search(r"SanctSound_([A-Za-z0-9]+_[A-Za-z0-9]+)_([^_]+)_\d{8}T\d{6}Z\.wav$", blob.name)
    if not match:
        continue

    site_id = match.group(1)
    event_name = match.group(2)

    if not any(k in event_name for k in keywords):
        continue
    
    combined_key = f"{event_name} - Soundscape {site_id}"

    public_url = f"https://storage.googleapis.com/{bucket_name}/{blob.name}"

    urls.setdefault(combined_key, []).append(public_url)

urls

{'bocaccio - Soundscape CI01_01': ['https://storage.googleapis.com/noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_01_sound_clips/data/SanctSound_CI01_01_bocaccio_20181101T100353Z.wav'],
 'snappingshrimp - Soundscape CI01_01': ['https://storage.googleapis.com/noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_01_sound_clips/data/SanctSound_CI01_01_snappingshrimp_20181101T080448Z.wav',
  'https://storage.googleapis.com/noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_01_sound_clips/data/SanctSound_CI01_01_snappingshrimp_20190207T221053Z.wav'],
 'plainfinmidshipman - Soundscape CI01_02': ['https://storage.googleapis.com/noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_02_sound_clips/data/SanctSound_CI01_02_plainfinmidshipman_20190426T085831Z.wav',
  'https://storage.googleapis.com/noaa-passive-bioacoustic/sanctsound/products/sound_clips/ci01/sanctsound_ci01_02_sound_clips/da

# Step 2
Function that will load audio for a given key.
My thought is user could "add" individual files they want to use to their library since it would probably take too long to store all in local memory (even though they'd be stored numerically)

In [19]:
def load_audio(key):
    if key not in urls:
        raise ValueError(f"Key '{key}' not found in urls.")

    # Each key may have multiple URLs; take the first one unless specified
    full_url = urls[key][0]

    response = requests.get(full_url)
    response.raise_for_status()

    audio_bytes = io.BytesIO(response.content)
    data, sr = sf.read(audio_bytes)

    return data, sr

load_audio("windwaves - Soundscape SB03_13")

(array([0.00000000e+00, 1.52587891e-04, 3.96728516e-04, ...,
        4.27246094e-04, 9.15527344e-05, 3.05175781e-05], shape=(240001,)),
 48000)